# 02 · 多头注意力与 Transformer 块  Multi-Head Attention & Transformer Block

> **能力标签**：GA1（Transformer 架构 / Transformer Architecture）

## 目标 Objectives

完成本课后，你应该能够：

1. 解释**多头注意力**（Multi-Head Attention, MHA）相比单头的优势：不同头可捕捉不同的关系模式。
2. 实现**分头**（head split）和**合并**（head merge）操作：$[B, T, D] \leftrightarrow [B, H, T, d_h]$，其中 $d_h = D/H$。
3. 写出完整 MHA 前向传播：$W_Q, W_K, W_V$ 投影 → 分头 → 缩放点积注意力 → 合并 → $W_O$ 输出投影。
4. 描述 **Transformer 块**（Transformer Block）的结构：MHA + FFN + 残差连接 + LayerNorm。
5. 实现并解释**正弦位置编码**（Sinusoidal Positional Encoding），理解其频率设计原理。

> 对应能力：**GA1**

## 概念讲解 Concepts

### 1. 多头注意力 Multi-Head Attention

单头注意力用一组 $Q, K, V$ 线性投影捕捉全局依赖。**多头**将模型维度 $D$ 切成 $H$ 份（每头维度 $d_h = D/H$），各头独立学习不同的注意力模式，再将结果拼接并经 $W_O$ 投影：

$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_H)W^O$$

实现上为了效率，投影矩阵 $W^Q, W^K, W^V$ 各为 $[D, D]$（包含所有头的参数），再通过 **reshape + transpose** 分头：

$$[B, T, D] \xrightarrow{\text{reshape}} [B, T, H, d_h] \xrightarrow{\text{transpose}} [B, H, T, d_h]$$

合并时反向操作：$[B, H, T, d_h] \xrightarrow{\text{transpose}} [B, T, H, d_h] \xrightarrow{\text{reshape}} [B, T, D]$

---

### 2. Transformer 块 Transformer Block

标准 Pre-LN Transformer 块的结构（以编码器为例）：

```
x → LayerNorm → MHA → + (残差 residual) → LayerNorm → FFN → + (残差)
```

$$x_1 = x + \text{MHA}(\text{LN}(x))$$
$$x_2 = x_1 + \text{FFN}(\text{LN}(x_1))$$

**FFN（前馈网络 Feed-Forward Network）**：两层线性 + ReLU/GELU，内部维度通常为 $4D$：

$$\text{FFN}(x) = W_2 \cdot \text{ReLU}(W_1 x + b_1) + b_2$$

**残差连接**（Residual Connection）缓解梯度消失；**LayerNorm** 稳定训练。

---

### 3. 正弦位置编码 Sinusoidal Positional Encoding

Transformer 本身对序列顺序不敏感，需显式注入位置信息。Vaswani et al. 提出：

$$PE_{(pos, 2i)} = \sin\!\left(\frac{pos}{10000^{2i/d}}\right), \quad PE_{(pos, 2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d}}\right)$$

- 每个位置 $pos$ 得到一个 $d$ 维向量；
- 不同维度使用不同频率，低维度频率高（短程），高维度频率低（长程）；
- 直接叠加到词嵌入上：$x' = x + PE$。

## 示例 Worked Example

In [ ]:
# ── 多头注意力完整实现（镜像 w7-mha）────────────────────────────────────────
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)


class MultiHeadAttention(nn.Module):
    """多头自注意力：投影 Q/K/V → 分头 → 缩放点积注意力 → 合并 → 输出投影。
    Multi-head self-attention with split/merge and output projection.
    """
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, "d_model 必须能被 n_heads 整除"
        self.n_heads = n_heads
        self.d_head = d_model // n_heads

        # 四个线性投影（无 bias 也可，这里与参考方案保持一致）
        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.wo = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, D = x.shape

        def split_heads(t):
            """[B,T,D] → [B,H,T,d_head]  (分头 Split into heads)"""
            return t.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        # ① 线性投影 Linear projections
        q = split_heads(self.wq(x))   # (B, H, T, d_head)
        k = split_heads(self.wk(x))
        v = split_heads(self.wv(x))

        # ② 缩放点积注意力 Scaled dot-product attention
        scores = (q @ k.transpose(-2, -1)) / (self.d_head ** 0.5)  # (B,H,T,T)
        attn = scores.softmax(dim=-1)

        # ③ 合并头 Merge heads: (B,H,T,d_head) → (B,T,D)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, D)

        # ④ 输出投影 Output projection
        return self.wo(out)


# ── 形状验证 Shape verification ───────────────────────────────────────────────
B, T, D, H = 2, 6, 32, 4
mha = MultiHeadAttention(d_model=D, n_heads=H)
x = torch.randn(B, T, D)
out = mha(x)

print("多头注意力前向传播 MHA forward pass:")
print(f"  输入 x.shape       = {x.shape}    (B={B}, T={T}, D={D})")
print(f"  输出 out.shape     = {out.shape}   (应与输入形状相同)")
print(f"  n_heads={H}, d_head={D//H}")
assert out.shape == (B, T, D)
print("✓ MHA 输出形状正确 MHA output shape OK")


In [ ]:
# ── 分头/合并操作可视化 Head Split/Merge Demo ──────────────────────────────
# 演示 [B,T,D] ↔ [B,H,T,d_head] 的变换过程

B, T, D, H = 1, 4, 8, 2
d_head = D // H

x_demo = torch.arange(B * T * D, dtype=torch.float).reshape(B, T, D)
print("原始张量 x_demo [B=1, T=4, D=8]:")
print(x_demo[0])   # (T, D)

# 分头 Split
split = x_demo.view(B, T, H, d_head).transpose(1, 2)  # (B, H, T, d_head)
print(f"\n分头后 After split [B=1, H=2, T=4, d_head=4]:")
print(f"  head 0 (前 4 维):\n{split[0, 0]}")
print(f"  head 1 (后 4 维):\n{split[0, 1]}")

# 合并 Merge
merged = split.transpose(1, 2).contiguous().view(B, T, D)
print(f"\n合并后 After merge [B=1, T=4, D=8]:")
print(merged[0])

assert torch.allclose(x_demo, merged)
print("✓ split → merge 无信息损失 Lossless round-trip OK")


In [ ]:
# ── 正弦位置编码 Sinusoidal Positional Encoding ───────────────────────────
import math

def sinusoidal_pe(T, d_model):
    """生成 Vaswani et al. 正弦位置编码矩阵 (T, d_model)。
    PE[pos, 2i]   = sin(pos / 10000^(2i/d))
    PE[pos, 2i+1] = cos(pos / 10000^(2i/d))
    """
    pe = torch.zeros(T, d_model)
    pos = torch.arange(T, dtype=torch.float).unsqueeze(1)   # (T, 1)
    i   = torch.arange(0, d_model, 2, dtype=torch.float)    # (d_model//2,)
    div = torch.exp(i * (-math.log(10000.0) / d_model))     # 数值稳定写法
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe


T_pe, D_pe = 50, 64
pe = sinusoidal_pe(T_pe, D_pe)
print(f"位置编码矩阵 PE shape: {pe.shape}  (T={T_pe}, d_model={D_pe})")
print(f"  第 0 个位置: {pe[0, :6].tolist()}")
print(f"  第 1 个位置: {pe[1, :6].tolist()}")

# 验证值域 in [-1, 1]
assert pe.abs().max() <= 1.0 + 1e-5
print("✓ 位置编码值域 ∈ [-1,1] Positional encoding range [-1,1] OK")


In [ ]:
# ── 完整 Transformer 块 Full Transformer Block ────────────────────────────
class FeedForward(nn.Module):
    """两层前馈网络（FFN），内部维度 4×d_model，ReLU 激活。
    Two-layer FFN: Linear → ReLU → Linear; inner dim = 4×d_model.
    """
    def __init__(self, d_model):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.ReLU(),
            nn.Linear(4 * d_model, d_model),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    """Pre-LN Transformer 块：MHA + FFN，各带残差连接与 LayerNorm。
    Pre-LN Transformer block: x → LN → MHA → residual → LN → FFN → residual.
    """
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.mha = MultiHeadAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model)

    def forward(self, x):
        x = x + self.mha(self.ln1(x))   # 残差 + MHA  (residual + MHA)
        x = x + self.ffn(self.ln2(x))   # 残差 + FFN  (residual + FFN)
        return x


torch.manual_seed(0)
B, T, D, H = 2, 8, 64, 8
block = TransformerBlock(d_model=D, n_heads=H)
x = torch.randn(B, T, D)
out = block(x)

print("Transformer 块前向传播 TransformerBlock forward:")
print(f"  输入 x.shape  = {x.shape}")
print(f"  输出 out.shape = {out.shape}  (应与输入形状相同)")
assert out.shape == (B, T, D)

# 参数数量
n_params = sum(p.numel() for p in block.parameters())
print(f"  参数量 Parameters: {n_params:,}")
print("✓ TransformerBlock 前向传播正确 OK")


In [ ]:
# ── 位置编码可视化 PE Heatmap ─────────────────────────────────────────────
import os
import tempfile
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

T_vis, D_vis = 60, 64
pe_vis = sinusoidal_pe(T_vis, D_vis)

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(pe_vis.T.numpy(), aspect="auto", cmap="RdYlBu", vmin=-1, vmax=1)
ax.set_xlabel("位置 Position", fontsize=10)
ax.set_ylabel("编码维度 Encoding Dim", fontsize=10)
ax.set_title("正弦位置编码热图 Sinusoidal Positional Encoding", fontsize=11)
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()

tmpdir = tempfile.mkdtemp()
fig_path = os.path.join(tmpdir, "pe_heatmap.png")
fig.savefig(fig_path, dpi=100, bbox_inches="tight")
plt.close(fig)
print(f"位置编码热图已保存 PE heatmap saved: {fig_path}")
print("  低维度（上方行）：高频振荡，捕捉短程位置关系")
print("  高维度（下方行）：低频振荡，捕捉长程位置关系")


## 动手 Exercise

In [ ]:
# ── 动手练习：为 MHA 添加因果掩码支持 Add Causal Mask to MHA ───────────────
# 任务：扩展 MultiHeadAttention，接受 mask 参数，并在注意力得分上应用掩码。
# 验证：对 T=4 的输入，上三角位置的注意力权重应为 0。

class MultiHeadAttentionMasked(nn.Module):
    """带因果掩码的多头注意力。Multi-head attention with optional causal mask."""
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.wo = nn.Linear(d_model, d_model)

    def forward(self, x, causal=False):
        B, T, D = x.shape

        def split(t):
            return t.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        q, k, v = split(self.wq(x)), split(self.wk(x)), split(self.wv(x))
        scores = (q @ k.transpose(-2, -1)) / (self.d_head ** 0.5)

        if causal:
            # 因果掩码：上三角置 -inf
            mask = torch.tril(torch.ones(T, T, device=x.device, dtype=torch.bool))
            scores = scores.masked_fill(~mask, float("-inf"))

        attn = scores.softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, D)
        return self.wo(out), attn


torch.manual_seed(2025)
B, T, D, H = 1, 4, 16, 2
mha_m = MultiHeadAttentionMasked(D, H)
x_ex = torch.randn(B, T, D)
out_ex, attn_ex = mha_m(x_ex, causal=True)

print(f"因果 MHA 输出形状: {out_ex.shape}  (应为 ({B}, {T}, {D}))")
print(f"注意力权重（第 0 个 batch，第 0 个 head）：")
print(attn_ex[0, 0].detach().round(decimals=3))

# 上三角（未来位置）应全为 0
upper_tri = attn_ex[0, :, torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)]
assert torch.allclose(upper_tri, torch.zeros_like(upper_tri), atol=1e-6)
print("✓ 因果掩码 MHA 正确：未来位置权重为 0 Causal MHA OK")


## 延伸阅读 Further Reading

1. **Vaswani et al. "Attention Is All You Need" (2017)**：<https://arxiv.org/abs/1706.03762> — 原始 Transformer 论文；第 3 节详述 MHA 与 PE 的设计。
2. **The Annotated Transformer (Harvard NLP)**：<https://nlp.seas.harvard.edu/annotated-transformer/> — 带逐行注释的 PyTorch 实现，与本课 MHA 实现直接对应。
3. **Karpathy "Let's build GPT" (YouTube)**：<https://www.youtube.com/watch?v=kCc8FmEb1nY> — 手写 nanoGPT，含 MHA、TransformerBlock 的完整实现与解释。
4. **Ba et al. "Layer Normalization" (2016)**：<https://arxiv.org/abs/1607.06450> — LayerNorm 的原始论文，解释其在 Transformer 中的作用。
5. **Illustrated Transformer (Jay Alammar)**：<https://jalammar.github.io/illustrated-transformer/> — 图解多头注意力分头/合并操作，适合概念入门。

## 关联练习 Related Assignments

```bash
python check.py 02-mha
```

> 实现 `MultiHeadAttention`（含 `wq/wk/wv/wo` 投影、分头、缩放点积注意力、合并头）。

> 能力标签：**GA1** · threshold ≥ 0.7